# Franchise Analytics

**Business question:** Where are our franchises concentrated, how are they distributed by size, and which markets have the deepest footprint?

This notebook analyses the bakehouse franchise network to answer:
- Which countries have the most franchise locations?
- How does franchise size distribution vary by market?
- Which markets are dominated by large vs. small outlets?
- How do countries rank by total franchise count?

**Output table:** `{catalog}.{schema}.franchise_analytics`  
**Source table:** `samples.bakehouse.sales_franchises`

In [ ]:
# Default catalog is the dev workspace — avoids accidentally writing to prod
# when running the notebook interactively.
dbutils.widgets.text("catalog", "dev_databricks_analytics")
dbutils.widgets.text("schema", "md")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE SCHEMA {schema}")

print(f"Target: {catalog}.{schema}")

## Step 1 — Explore raw franchise data

Preview the schema and a sample of rows to understand the structure
before writing any transformation logic.

In [ ]:
franchises = spark.read.table("samples.bakehouse.sales_franchises")
print(f"Franchises: {franchises.count():,} locations")
franchises.printSchema()
franchises.show(10, truncate=False)

## Step 2 — Geographic footprint

Count franchises per country and rank them. `DENSE_RANK()` is used so tied
countries share a rank and no gaps appear in the ranking sequence.

In [ ]:
geo_footprint = spark.sql("""
    SELECT
        country,
        COUNT(*)                                              AS total_franchises,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)   AS pct_of_global_network,
        DENSE_RANK() OVER (ORDER BY COUNT(*) DESC)            AS country_rank
    FROM samples.bakehouse.sales_franchises
    GROUP BY country
    ORDER BY country_rank
""")
geo_footprint.show(truncate=False)

## Step 3 — Size distribution by country

Breaking down by size reveals whether a market is served primarily by
large flagship stores or smaller local outlets — which has implications
for staffing, inventory, and marketing budget allocation.

In [ ]:
size_by_country = spark.sql("""
    SELECT
        country,
        size,
        COUNT(*)                                              AS franchise_count,
        -- Rank sizes within each country to find dominant format per market
        DENSE_RANK() OVER (
            PARTITION BY country
            ORDER BY COUNT(*) DESC
        )                                                     AS size_rank_within_country
    FROM samples.bakehouse.sales_franchises
    GROUP BY country, size
    ORDER BY country, size_rank_within_country
""")
size_by_country.show(50, truncate=False)

## Step 4 — Build the consolidated franchise analytics table

Combine geographic footprint with size breakdown into one output.
Each row represents a unique (country, size) combination with its
full context: count, percentage of that country's network, and percentage
of the global network.

In [ ]:
franchise_analytics = spark.sql("""
    WITH country_totals AS (
        SELECT country, COUNT(*) AS country_total
        FROM samples.bakehouse.sales_franchises
        GROUP BY country
    ),
    global_total AS (
        SELECT COUNT(*) AS grand_total
        FROM samples.bakehouse.sales_franchises
    )
    SELECT
        f.country,
        f.size,
        COUNT(*)                                              AS number_of_franchises,
        ct.country_total                                      AS country_total_franchises,
        -- Share of this size within its country
        ROUND(COUNT(*) * 100.0 / NULLIF(ct.country_total, 0), 2)
                                                              AS pct_within_country,
        -- Share of global network
        ROUND(COUNT(*) * 100.0 / NULLIF(gt.grand_total, 0), 2)
                                                              AS pct_of_global_network,
        -- Rank countries globally by their total franchise count
        DENSE_RANK() OVER (ORDER BY ct.country_total DESC)    AS country_rank
    FROM samples.bakehouse.sales_franchises  f
    JOIN country_totals  ct  ON f.country = ct.country
    CROSS JOIN global_total  gt
    GROUP BY f.country, f.size, ct.country_total, gt.grand_total
    ORDER BY country_rank, number_of_franchises DESC
""")
franchise_analytics.show(50, truncate=False)

## Step 5 — Write to Unity Catalog

In [ ]:
franchise_analytics.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.franchise_analytics")
print(f"Written {franchise_analytics.count()} rows to {catalog}.{schema}.franchise_analytics")